In [1]:
import openpyxl
import pyomo.environ as pe
import numpy as np
import os
import pandas as pd
from datetime import datetime, timedelta

In [2]:
Year=2019
Month=2
Day=14
T=60
N_DEV = 10


T_ON = [[0 for j in range(10)] for i in range(T)] # Status of appliance j at time i
T_OFF = [[0 for j in range(10)] for i in range(T)] # Status of appliance j at time 
X_aux = [[0 for j in range(10)] for i in range(T)] # Status of appliance j at time i

# Especifique o diretório onde estão seus arquivos Excel.
diretorio = '/Users/luismartins/Desktop/Tese/NexIK_Flexibility/Data_Valores Reais'

# Lista todos os arquivos na pasta.
arquivos_na_pasta = os.listdir(diretorio)

In [3]:
# Rolling Windows
WINDOW_MINUTES = 60   # cada otimização olha 1 hora
STEP_MINUTES = 60     # avança 1 hora de cada vez
DAY_MINUTES = 24 * 60

# instante inicial e final do dia em análise
day_start_dt = datetime(Year, Month, Day, 0, 0, 0)
day_end_dt = day_start_dt + timedelta(days=1)

# =========================
# DEVICE GROUPS
# =========================
BUFFERED_DEVICES = {0, 2}        # Drinks Refrigerator, Garde Manger
OPERATIONAL_DEVICES = {1, 3, 4, 5, 6}  # Blast Chiller, Ice Machine, Dual Fryer, Salamander, Infrared Shelf
NON_FLEXIBLE_DEVICES = {7, 8, 9}  # Glass Washer, Dish Washer, Convection Oven

N_MIN_ACTIVATIONS = 2
T_NO_ACTIVITY_FALLBACK = 500
T_OFF_FORCED_ON = 5
T_MIN = 5

# diretório dos ficheiros
diretorio = '/Users/luismartins/Desktop/Tese/NexIK_Flexibility/Data_Valores Reais'

# mapeamento appliance -> ficheiro de ativações
activation_files = {
    0: "activations_Drinks_Refrigerator.csv",
    1: "activations_Blast_Chiller.csv",
    2: "activations_Garde_Manger_10.csv",
    3: "activations_Ice_Machine.csv",
    4: "activations_Dual_Fryer.csv",
    5: "activations_Salamander_1.csv",
    6: "activations_Infrared_Shelf.csv",
    7: "activations_Glass_Washer.csv",
    8: "activations_Dish_Washer.csv",
    9: "activations_Convection_Oven_1.csv",
}

# lista das 24 janelas rolling horizon
rolling_windows = []

current_start = day_start_dt
while current_start < day_end_dt:
    current_end = current_start + timedelta(minutes=WINDOW_MINUTES)
    rolling_windows.append((current_start, current_end))
    current_start += timedelta(minutes=STEP_MINUTES)



print(f"Janela 1: {rolling_windows[0][0]} -> {rolling_windows[0][1]}")

Janela 1: 2019-02-14 00:00:00 -> 2019-02-14 01:00:00


In [4]:
def minute_of_day(ts):
    m = ts.hour * 60 + ts.minute
    if ts.second >= 30:
        m += 1
    return min(m, 1439)


def get_time_zone(minute):
    """
    0 -> 00:00 - 09:59
    1 -> 10:00 - 19:29
    2 -> 19:30 - 22:29
    3 -> 22:30 - 23:59
    """
    if 0 <= minute <= 599:
        return 0
    elif 600 <= minute <= 1169:
        return 1
    elif 1170 <= minute <= 1349:
        return 2
    else:
        return 3


def load_activation_day_csv(diretorio, nome_do_arquivo, target_date):
    caminho_arquivo = os.path.join(diretorio, nome_do_arquivo)
    if not os.path.exists(caminho_arquivo):
        raise FileNotFoundError(f"File not found: {caminho_arquivo}")

    df = pd.read_csv(caminho_arquivo, sep=';').iloc[:, 0:3].copy()

    df["first_min_on"] = pd.to_datetime(df["first_min_on"], utc=True, errors="coerce")
    df["last_min_on"]  = pd.to_datetime(df["last_min_on"],  utc=True, errors="coerce")

    df["first_min_on"] = df["first_min_on"].dt.tz_convert(None)
    df["last_min_on"]  = df["last_min_on"].dt.tz_convert(None)

    df = df.dropna(subset=["first_min_on", "last_min_on"]).sort_values("first_min_on").reset_index(drop=True)

    target_date = pd.to_datetime(target_date).date()
    df_day = df[df["first_min_on"].dt.date == target_date].copy().reset_index(drop=True)

    return df_day


def build_X_day_for_device(j, nome_do_arquivo, diretorio, target_date, X_day):
    """
    Constrói X_day[:, j] para o dia inteiro.
    """
    X_day[:, j] = 0

    df_day = load_activation_day_csv(diretorio, nome_do_arquivo, target_date)

    if df_day.empty:
        return

    for _, row in df_day.iterrows():
        start = minute_of_day(row["first_min_on"])
        end   = minute_of_day(row["last_min_on"])

        if end >= start:
            X_day[start:end + 1, j] = 1
        else:
            X_day[start:1440, j] = 1
            X_day[0:end + 1, j] = 1


def compute_operational_ton_toff_day(j, nome_do_arquivo, diretorio, target_date):
    """
    Operational devices:
    T_ON/T_OFF por timezone, avaliando presença ON dentro da timezone,
    mesmo quando a ativação começou antes da timezone.

    Regras:
      - se não há minutos ON na timezone:
            T_ON  = T_NO_ACTIVITY_FALLBACK
            T_OFF = T_NO_ACTIVITY_FALLBACK

      - se há minutos ON na timezone:
            T_ON = duração total ON dentro da timezone, ou mediana dos blocos ON na timezone

            se houver gaps OFF observados:
                T_OFF = mediana dos gaps OFF
            senão:
                T_OFF = T_OFF_FORCED_ON
    """

    T_ON_day_j = np.full(DAY_MINUTES, T_NO_ACTIVITY_FALLBACK, dtype=int)
    T_OFF_day_j = np.full(DAY_MINUTES, T_NO_ACTIVITY_FALLBACK, dtype=int)

    df_day = load_activation_day_csv(diretorio, nome_do_arquivo, target_date)

    if df_day.empty:
        return T_ON_day_j, T_OFF_day_j

    # construir X_day do device
    X_dev = np.zeros(DAY_MINUTES, dtype=int)

    for _, row in df_day.iterrows():
        start = minute_of_day(row["first_min_on"])
        end = minute_of_day(row["last_min_on"])

        if end >= start:
            X_dev[start:end + 1] = 1
        else:
            X_dev[start:DAY_MINUTES] = 1
            X_dev[0:end + 1] = 1

    zone_limits = {
        0: (0, 600),       # 00:00 - 09:59
        1: (600, 1170),   # 10:00 - 19:29
        2: (1170, 1350),  # 19:30 - 22:29
        3: (1350, 1440),  # 22:30 - 23:59
    }

    zone_ton = {}
    zone_toff = {}

    for z, (start_z, end_z) in zone_limits.items():
        X_zone = X_dev[start_z:end_z]

        on_minutes = int(np.sum(X_zone))

        # sem presença ON na timezone
        if on_minutes == 0:
            zone_ton[z] = T_NO_ACTIVITY_FALLBACK
            zone_toff[z] = T_NO_ACTIVITY_FALLBACK
            continue

        # há presença ON na timezone
        # calcular blocos ON e OFF dentro da timezone
        on_blocks = []
        off_blocks = []

        current_state = X_zone[0]
        count = 1

        for val in X_zone[1:]:
            if val == current_state:
                count += 1
            else:
                if current_state == 1:
                    on_blocks.append(count)
                else:
                    off_blocks.append(count)

                current_state = val
                count = 1

        # fechar último bloco
        if current_state == 1:
            on_blocks.append(count)
        else:
            off_blocks.append(count)

        # T_ON: se há blocos ON, usa mediana dos blocos ON
        if len(on_blocks) > 0:
            ton_z = max(1, int(np.median(on_blocks)))
        else:
            ton_z = T_NO_ACTIVITY_FALLBACK

        # T_OFF:
        # só usar gaps OFF se existirem dentro da zona
        # se está sempre ON ou sem OFF observado, usar forced-on
        valid_off_blocks = [b for b in off_blocks if b > 0]

        if len(valid_off_blocks) > 0:
            toff_z = max(1, int(np.median(valid_off_blocks)))
        else:
            toff_z = T_OFF_FORCED_ON

        zone_ton[z] = ton_z
        zone_toff[z] = toff_z

    for t in range(DAY_MINUTES):
        z = get_time_zone(t)
        T_ON_day_j[t] = zone_ton[z]
        T_OFF_day_j[t] = zone_toff[z]
        
    return T_ON_day_j, T_OFF_day_j


def compute_buffered_ton_toff_for_window(j, X_window):
    """
    Buffered devices:
    Replica a lógica da Ana dentro da janela de 60 min.
    """
    T_ON_window_j = np.zeros(T, dtype=int)
    T_OFF_window_j = np.zeros(T, dtype=int)

    cont_on = 0
    cont_off = 0
    periodo = 0
    first = 0

    for i in range(T):
        if i == 0 and X_window[i] == 0:
            first = 0

        if X_window[i] == 0:
            cont_off += 1

        if X_window[i] == 1:
            cont_on += 1

        if X_window[i] == 1 and i < T - 1 and X_window[i + 1] == 0:
            periodo = i
            for k in range(first, periodo + 1):
                T_OFF_window_j[k] = max(1, cont_off)
                T_ON_window_j[k] = max(1, cont_on)

            cont_on = 1
            cont_off = 1
            first = periodo + 1

        elif X_window[i] == 1 and i == T - 1:
            for k in range(first, T):
                T_OFF_window_j[k] = max(1, cont_off)
                T_ON_window_j[k] = max(1, cont_on)

        else:
            if i == T - 1:
                for k in range(first, T):
                    if X_window[i] == 1:
                        T_ON_window_j[k] = max(1, cont_on)
                        T_OFF_window_j[k] = max(1, T_ON_window_j[k])
                    else:
                        T_OFF_window_j[k] = max(1, cont_off - 1)
                        T_ON_window_j[k] = max(1, T_OFF_window_j[k])

    T_ON_window_j[T_ON_window_j <= 0] = 1
    T_OFF_window_j[T_OFF_window_j <= 0] = 1

    return T_ON_window_j, T_OFF_window_j


def build_X_day_all_devices(day_start_dt, activation_files, diretorio):
    X_day = np.zeros((DAY_MINUTES, N_DEV), dtype=int)
    target_date = day_start_dt.date()

    for j in range(N_DEV):
        build_X_day_for_device(
            j=j,
            nome_do_arquivo=activation_files[j],
            diretorio=diretorio,
            target_date=target_date,
            X_day=X_day
        )

    return X_day

In [5]:
N_WINDOWS = len(rolling_windows)

# 1) Construir apenas X_day global
X_day = build_X_day_all_devices(
    day_start_dt=day_start_dt,
    activation_files=activation_files,
    diretorio=diretorio
)

# 2) Inicializar matrizes por janela
X_onoff_FC_matrix = np.zeros((N_WINDOWS, T, N_DEV), dtype=int)
T_ON_FC_matrix    = np.ones((N_WINDOWS, T, N_DEV), dtype=int)
T_OFF_FC_matrix   = np.ones((N_WINDOWS, T, N_DEV), dtype=int)

target_date = day_start_dt.date()

# 3) Preencher janela a janela
for w_idx, (window_start_dt, window_end_dt) in enumerate(rolling_windows):
    start_minute = int((window_start_dt - day_start_dt).total_seconds() / 60)
    end_minute = start_minute + T

    # X vem sempre do dia global
    X_window = X_day[start_minute:end_minute, :]
    X_onoff_FC_matrix[w_idx, :, :] = X_window

    for j in range(N_DEV):
        nome_do_arquivo = activation_files[j]

        # Buffered devices -> lógica da Ana por janela
        if j in BUFFERED_DEVICES:
            T_ON_j, T_OFF_j = compute_buffered_ton_toff_for_window(j, X_window[:, j])

        # Operational devices -> timezone + fallback conservador
        elif j in OPERATIONAL_DEVICES:
            T_ON_day_j, T_OFF_day_j = compute_operational_ton_toff_day(
                j=j,
                nome_do_arquivo=nome_do_arquivo,
                diretorio=diretorio,
                target_date=target_date
            )
            T_ON_j = T_ON_day_j[start_minute:end_minute]
            T_OFF_j = T_OFF_day_j[start_minute:end_minute]

        # Non-flexible -> dummy
        else:
            T_ON_j = np.ones(T, dtype=int)
            T_OFF_j = np.ones(T, dtype=int)

        T_ON_FC_matrix[w_idx, :, j] = T_ON_j
        T_OFF_FC_matrix[w_idx, :, j] = T_OFF_j

In [6]:
for j in BUFFERED_DEVICES:
    print(f"\nBuffered device {j} - first windows")
    print("Window 1 T_ON :", T_ON_FC_matrix[13, :, j])
    print("Window 1 T_OFF:", T_OFF_FC_matrix[13, :, j])

for j in OPERATIONAL_DEVICES:
    print(f"\nOperational device {j} - first windows")
    print("Window 1 T_ON :", T_ON_FC_matrix[13, :, j])
    print("Window 1 T_OFF:", T_OFF_FC_matrix[13, :, j])


Buffered device 0 - first windows
Window 1 T_ON : [30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30
 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 30 14 14
 14 14 14 14 14 14 14 14 14 14 14 14]
Window 1 T_OFF: [16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 16
 16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 14 14
 14 14 14 14 14 14 14 14 14 14 14 14]

Buffered device 2 - first windows
Window 1 T_ON : [ 5  5  5  5  5  5  5  5  6  6  6  6  6  6  6  6  6  6  6  6  6  6  6  6
  6  6  6  6  6  6  6  6  6  6  6  6  6 23 23 23 23 23 23 23 23 23 23 23
 23 23 23 23 23 23 23 23 23 23 23 23]
Window 1 T_OFF: [ 3  3  3  3  3  3  3  3 25 25 25 25 25 25 25 25 25 25 25 25 25 25 25 25
 25 25 25 25 25 25 25 25 25 25 25 25 25 23 23 23 23 23 23 23 23 23 23 23
 23 23 23 23 23 23 23 23 23 23 23 23]

Operational device 1 - first windows
Window 1 T_ON : [3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3
 

In [7]:
# ======================================
# LEITURA DO INFO.XLSX
# ======================================

df_info = pd.read_excel('/Users/luismartins/Desktop/Tese/NexIK_Flexibility/Forecast_24h/Real Values - FC24 Cost/Info.xlsx')

print("Colunas do Info.xlsx:")
print(df_info.columns.tolist())

# se quiseres confirmar o número de equipamentos do Excel
print("\nNúmero de linhas no Info.xlsx:", len(df_info))

# opcional: garantir consistência
if len(df_info) != N_DEV:
    raise ValueError(f"N_DEV={N_DEV}, mas o Info.xlsx tem {len(df_info)} linhas.")

# parâmetros por equipamento
dev_names = df_info["Dev"].tolist()

# índice do tipo de flexibilidade
Type = df_info["Type"].to_numpy(dtype=int)

# estado inicial
# para já vamos ler do Excel; mais tarde podes forçar a zero ou atualizar dinamicamente
State_Init = df_info["State_Init"].to_numpy(dtype=int)

# potência máxima
Power_max = df_info["Power_max"].to_numpy(dtype=float)

# pesos da componente de potência
Weight_P_max = df_info["Weight_P_max"].to_numpy(dtype=float)
Weight_P_min = df_info["Weight_P_min"].to_numpy(dtype=float)

# percentagens de potência
# MUITO IMPORTANTE: converter de % para fração
Y_max = df_info["Y_max (%)"].to_numpy(dtype=float) / 100.0
Y_min = df_info["Y_min (%)"].to_numpy(dtype=float) / 100.0

# tempo mínimo para manter alteração de potência
TimeHoldPower = df_info["TimeHoldPower"].to_numpy(dtype=float)

# impressão de controlo
print("\nResumo dos equipamentos:")
for j in range(N_DEV):
    print(
        f"{j}: {dev_names[j]} | "
        f"Type={Type[j]} | "
        f"State_Init={State_Init[j]} | "
        f"Pmax={Power_max[j]} | "
        f"Ymin={Y_min[j]} | "
        f"Ymax={Y_max[j]}"
    )

Colunas do Info.xlsx:
['Dev', 'j', 'Type', 'State_Init', 'Power_max', 'Weight_P_max', 'Weight_P_min', 'Y_max (%)', 'Y_min (%)', 'TimeHoldPower']

Número de linhas no Info.xlsx: 10

Resumo dos equipamentos:
0: Drinks Refrigerator | Type=2 | State_Init=0 | Pmax=5000.0 | Ymin=1.0 | Ymax=1.0
1: Blast Chiller | Type=2 | State_Init=0 | Pmax=1500.0 | Ymin=1.0 | Ymax=1.0
2: Garde Manger | Type=2 | State_Init=0 | Pmax=1000.0 | Ymin=1.0 | Ymax=1.0
3: Ice Machine | Type=2 | State_Init=0 | Pmax=400.0 | Ymin=1.0 | Ymax=1.0
4: Dual Fryer | Type=2 | State_Init=0 | Pmax=5000.0 | Ymin=1.0 | Ymax=1.0
5: Salamander | Type=2 | State_Init=0 | Pmax=3000.0 | Ymin=1.0 | Ymax=1.0
6: Infrared Shelf | Type=2 | State_Init=0 | Pmax=800.0 | Ymin=1.0 | Ymax=1.0
7: Glass Washer | Type=0 | State_Init=0 | Pmax=1600.0 | Ymin=1.0 | Ymax=1.0
8: Dish Washer | Type=0 | State_Init=0 | Pmax=4200.0 | Ymin=1.0 | Ymax=1.0
9: Convection Oven | Type=0 | State_Init=0 | Pmax=7000.0 | Ymin=1.0 | Ymax=1.0


In [8]:
def price_function(Controlo):
    if Controlo == 0:
        Energy_price = [0.13] * T

    elif Controlo == 1:
        Energy_price = [0] * T
        price_variation = 30
        counter = 0
        for i in range(T):
            if counter < price_variation:
                Energy_price[i] = 0.06
            else:
                Energy_price[i] = 0.13
            counter += 1

    return Energy_price

def get_dynamic_threshold(minute):
    """
    minute: 0..1439
    """
    if 0 <= minute <= 599:        # 00:00 - 09:59
        return 5000.0
    elif 600 <= minute <= 1169:   # 10:00 - 19:29
        return 15000.0
    elif 1170 <= minute <= 1349:  # 19:30 - 22:29
        return 25000.0
    else:                         # 22:30 - 23:59
        return 10000.0
def function_weight(
    X_onoff_FC_matrix,
    T_ON_window,
    T_OFF_window,
    Type,
    State_Init_window
):
    W_time = np.zeros((T, N_DEV))

    for j in range(N_DEV):
        # Non-flexible e power-flexible
        if Type[j] == 0 or Type[j] == 1:
            W_time[:, j] = 10
            continue

        # Time-flexible
        if Type[j] == 2:
            # i = 0 -> reset por janela
            if State_Init_window[j] == 1:
                W_time[0, j] = 10 - 9 / T_ON_window[0, j]
            else:
                W_time[0, j] = 1 + 9 / T_OFF_window[0, j]

            W_time[0, j] = max(1.0, min(10.0, W_time[0, j]))

            # i > 0
            for i in range(1, T):
                if X_onoff_FC_matrix[i, j] == 1:
                    W_time[i, j] = W_time[i - 1, j] - 9 / T_ON_window[i, j]
                else:
                    W_time[i, j] = W_time[i - 1, j] + 9 / T_OFF_window[i, j]

                W_time[i, j] = max(1.0, min(10.0, W_time[i, j]))

    return W_time

In [9]:
def build_and_solve_window_model(
    X_forecast,
    T_ON_forecast,
    T_OFF_forecast,
    price_forecast,
    State_Init_window,
    P_lim,
    tee
):
    model = pe.ConcreteModel()

    # Sets
    model.loads = pe.Set(initialize=np.arange(N_DEV))
    model.Time = pe.Set(initialize=np.arange(T))

    # Params
    model.P_lim = pe.Param(initialize=P_lim)
    model.Type = pe.Param(model.loads, initialize=Type)
    model.P_max = pe.Param(model.loads, initialize=Power_max)

    model.State_init = pe.Param(
        model.loads,
        initialize=lambda m, j: int(State_Init_window[j])
    )

    model.T_OFF = pe.Param(
        model.Time, model.loads,
        initialize=lambda m, i, j: max(1, int(T_OFF_forecast[i][j]))
    )

    model.T_ON = pe.Param(
        model.Time, model.loads,
        initialize=lambda m, i, j: max(1, int(T_ON_forecast[i][j]))
    )

    model.W_max = pe.Param(model.loads, initialize=Weight_P_max)
    model.W_min = pe.Param(model.loads, initialize=Weight_P_min)
    model.Y_max = pe.Param(model.loads, initialize=Y_max)
    model.Y_min = pe.Param(model.loads, initialize=Y_min)

    model.X = pe.Param(
        model.Time, model.loads,
        initialize=lambda m, i, j: int(X_forecast[i][j])
    )

    model.price = pe.Param(
        model.Time,
        initialize=lambda m, i: float(price_forecast[i])
    )

    # Vars
    model.X_opt = pe.Var(model.Time, model.loads, domain=pe.Binary, initialize=0)
    model.W_time_opt = pe.Var(model.Time, model.loads, domain=pe.NonNegativeReals, initialize=1)
    model.P_rlx = pe.Var(model.Time, domain=pe.NonNegativeReals, initialize=0)
    model.power = pe.Var(model.Time, domain=pe.NonNegativeReals, initialize=0)

    # -------------------------
    # Constraints
    # -------------------------

    # Power limit
    def constraint_one(m, i):
        return sum(m.P_max[j] * m.X_opt[i, j] * m.Y_min[j] for j in m.loads) <= m.P_lim + m.P_rlx[i]
    model.constraint_one = pe.Constraint(model.Time, rule=constraint_one)

    # Non-flexible / power-flexible state
    def constraint_two(m, i, j):
        if m.Type[j] == 0 or m.Type[j] == 1:
            if i == 0:
                return m.X_opt[i, j] == m.State_init[j]
            else:
                return m.X_opt[i, j] == m.X[i, j]
        return pe.Constraint.Feasible
    model.constraint_two = pe.Constraint(model.Time, model.loads, rule=constraint_two)

    # Upper bound on time weight
    def constraint_three(m, i, j):
        if m.Type[j] == 2:
            return m.W_time_opt[i, j] <= 10
        return pe.Constraint.Feasible
    model.constraint_three = pe.Constraint(model.Time, model.loads, rule=constraint_three)

    # Lower bound on time weight
    def constraint_four(m, i, j):
        if m.Type[j] == 2:
            return m.W_time_opt[i, j] >= 1
        return pe.Constraint.Feasible
    model.constraint_four = pe.Constraint(model.Time, model.loads, rule=constraint_four)

    # Time-weight evolution
    def constraint_five(m, i, j):
        # Non-flexible + power-flexible
        if m.Type[j] == 0 or m.Type[j] == 1:
            return m.W_time_opt[i, j] == 10

        # Time-flexible
        elif m.Type[j] == 2:
            if i == 0:
                # reset por janela, coerente com function_weight
                return m.W_time_opt[i, j] == (
                    (1 + 9 / m.T_OFF[i, j]) * (1 - m.X_opt[i, j]) +
                    (10 - 9 / m.T_ON[i, j]) * m.X_opt[i, j]
                )
            else:
                return m.W_time_opt[i, j] == (
                    (m.W_time_opt[i - 1, j] - 9 / m.T_ON[i, j]) * m.X_opt[i, j] +
                    (m.W_time_opt[i - 1, j] + 9 / m.T_OFF[i, j]) * (1 - m.X_opt[i, j])
                )

        return pe.Constraint.Skip
    model.constraint_five = pe.Constraint(model.Time, model.loads, rule=constraint_five)

    # Power variable
    def constraint_power(m, i):
        return m.power[i] == sum(m.P_max[j] * m.X_opt[i, j] * m.Y_min[j] for j in m.loads)
    model.constraint_power = pe.Constraint(model.Time, rule=constraint_power)

    # -------------------------
    # Objective
    # -------------------------
    def objective_function(m):
        return sum(m.power[i] * m.price[i] + 10000 * m.P_rlx[i] for i in m.Time)
    model.objective = pe.Objective(rule=objective_function, sense=pe.minimize)

    # Solve
    solver = pe.SolverFactory('scip')
    solver.options["limits/time"] = 30  # seconds
    results = solver.solve(model, tee=tee)

    return model, results

In [10]:
def run_rolling_horizon_day(Controlo, P_lim, tee=True):
    rolling_results = []
    previous_final_state = None
    previous_final_weight = None
    
    X_matrix = np.zeros((N_WINDOWS, T, N_DEV))
    X_opt_matrix = np.zeros((N_WINDOWS, T, N_DEV))
    W_Time_opt_matrix = np.zeros((N_WINDOWS, T, N_DEV))
    W_base_matrix = np.zeros((N_WINDOWS, T, N_DEV))

    P_matrix = np.zeros((N_WINDOWS, T, N_DEV))
    P_sum_matrix = np.zeros((N_WINDOWS, T))

    P_opt_matrix = np.zeros((N_WINDOWS, T, N_DEV))
    P_sum_opt_matrix = np.zeros((N_WINDOWS, T))

    P_rlx_matrix = np.zeros((N_WINDOWS, T))
    OF_vector = np.zeros(N_WINDOWS)
    
    Cost_matrix = np.zeros((N_WINDOWS, T))
    Cost_opt_matrix = np.zeros((N_WINDOWS, T))
    price_matrix = np.zeros((N_WINDOWS, T))


    for w_idx, (window_start_dt, window_end_dt) in enumerate(rolling_windows):
        print(f"\n================ WINDOW {w_idx+1}/{len(rolling_windows)} ================")
        print(f"Window: {window_start_dt} -> {window_end_dt}")
        
        print("\nForecast X: ")       
        for i in range(T):
            print(X_onoff_FC_matrix[w_idx, i])
        
        X_forecast = X_onoff_FC_matrix[w_idx].copy()
        T_ON_forecast = T_ON_FC_matrix[w_idx].copy()
        T_OFF_forecast = T_OFF_FC_matrix[w_idx].copy()
        price_forecast = price_function(Controlo)
        price_matrix[w_idx, :] = price_forecast

        T_ON_forecast[T_ON_forecast <= 0] = 1
        T_OFF_forecast[T_OFF_forecast <= 0] = 1
        
        if w_idx == 0:
            State_Init_window = np.array(State_Init, dtype=int)
        else:
            State_Init_window = previous_final_state.copy()

        W_base = function_weight(
            X_onoff_FC_matrix=X_forecast,
            T_ON_window=T_ON_forecast,
            T_OFF_window=T_OFF_forecast,
            Type=Type,
            State_Init_window=State_Init_window,
        )
        W_base_matrix[w_idx, :, :] = W_base

        model, results = build_and_solve_window_model(
        X_forecast,
        T_ON_forecast,
        T_OFF_forecast,
        price_forecast,
        State_Init_window,
        P_lim,
        tee
        )

        print("\n-----------------------------Model Results--------------------------------------------\n")

        for i in model.Time:
            for j in range(N_DEV):
                X_matrix[w_idx, i, j] = pe.value(model.X[i, j])
                X_opt_matrix[w_idx, i, j] = round(pe.value(model.X_opt[i, j]))
                W_Time_opt_matrix[w_idx, i, j] = round(pe.value(model.W_time_opt[i, j]), 2)

        OF_vector[w_idx] = pe.value(model.objective)

        for i in range(T):
            for j in range(N_DEV):
                P_matrix[w_idx, i, j] = Power_max[j] * Y_min[j] * X_matrix[w_idx, i, j]
            P_sum_matrix[w_idx, i] = np.sum(P_matrix[w_idx, i, :])

        for i in range(T):
            for j in range(N_DEV):
                P_opt_matrix[w_idx, i, j] = Power_max[j] * Y_min[j] * X_opt_matrix[w_idx, i, j]
            P_sum_opt_matrix[w_idx, i] = np.sum(P_opt_matrix[w_idx, i, :])

        for i in model.Time:
            P_rlx_matrix[w_idx, i] = round(pe.value(model.P_rlx[i]), 2)

        P_matrix[w_idx] = np.round(P_matrix[w_idx], 2)
        P_sum_matrix[w_idx] = np.round(P_sum_matrix[w_idx], 2)
        P_opt_matrix[w_idx] = np.round(P_opt_matrix[w_idx], 2)
        P_sum_opt_matrix[w_idx] = np.round(P_sum_opt_matrix[w_idx], 2)
        
        Cost_matrix[w_idx, :] = np.round((P_sum_matrix[w_idx, :]/ 1000) * price_matrix[w_idx, :] * (1/60), 4)
        Cost_opt_matrix[w_idx, :] = np.round((P_sum_opt_matrix[w_idx, :] / 1000) * price_matrix[w_idx, :] * (1/60), 4)         


        # herdar para a janela seguinte, só o estado
        previous_final_state = X_opt_matrix[w_idx, -1, :].copy()

    return (
        X_matrix,
        X_opt_matrix,
        W_Time_opt_matrix,
        P_matrix,
        P_sum_matrix,
        P_opt_matrix,
        P_sum_opt_matrix,
        P_rlx_matrix,
        OF_vector,
        W_base_matrix,
        Cost_matrix,
        Cost_opt_matrix
    )

In [11]:
P_lim = 25000
Controlo = 1
(
    X_matrix,
    X_opt_matrix,
    W_Time_opt_matrix,
    P_matrix,
    P_sum_matrix,
    P_opt_matrix,
    P_sum_opt_matrix,
    P_rlx_matrix,
    OF_vector,
    W_base_matrix, 
    Cost_matrix,
    Cost_opt_matrix
) = run_rolling_horizon_day(Controlo, P_lim, tee=True)


================ WINDOW 1/24 ================
Window: 2019-02-14 00:00:00 -> 2019-02-14 01:00:00

Forecast X: 
[0 0 0 1 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0]
[0 0 1 1 0 0 0 0 0 0]
[0 0 1 1 0 0 0 0 0 0]
[0 0 1 1 0 0 0 0 0 0]
[0 0 1 1 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0]
[0 0 0 1

In [12]:
DAY_MINUTES = N_WINDOWS * T  # normalmente 24*60

# reshape tudo
X_day = X_matrix.reshape(DAY_MINUTES, N_DEV)
X_opt_day = X_opt_matrix.reshape(DAY_MINUTES, N_DEV)
W_day = W_Time_opt_matrix.reshape(DAY_MINUTES, N_DEV)
W_base_day = W_base_matrix.reshape(DAY_MINUTES, N_DEV)

P_sum_day = P_sum_matrix.reshape(DAY_MINUTES)
P_opt_sum_day = P_sum_opt_matrix.reshape(DAY_MINUTES)
P_rlx_day = P_rlx_matrix.reshape(DAY_MINUTES)

Cost_day = Cost_matrix.reshape(DAY_MINUTES)
Cost_opt_day = Cost_opt_matrix.reshape(DAY_MINUTES)

timestamps = [day_start_dt + timedelta(minutes=i) for i in range(DAY_MINUTES)]
timestamps_str = [ts.strftime("%H:%M") for ts in timestamps]

df_X = pd.DataFrame(X_day, columns=dev_names)

df_X.insert(0, "timestamp", timestamps_str)
df_X["P_sum"] = P_sum_day
df_X["P_opt_sum"] = P_opt_sum_day
df_X["P_rlx"] = P_rlx_day
df_X["Cost"] = Cost_day
df_X["Cost_opt"] = Cost_opt_day
df_X_opt = pd.DataFrame(X_opt_day, columns=dev_names)

df_X_opt.insert(0, "timestamp", timestamps_str)
df_X_opt["P_sum"] = P_sum_day
df_X_opt["P_opt_sum"] = P_opt_sum_day
df_X_opt["P_rlx"] = P_rlx_day
df_X_opt["Cost"] = Cost_day
df_X_opt["Cost_opt"] = Cost_opt_day

df_W = pd.DataFrame(W_day, columns=dev_names)
df_W.insert(0, "timestamp", timestamps_str)

df_W_base = pd.DataFrame(W_base_day, columns=dev_names)
df_W_base.insert(0, "timestamp", timestamps_str)

output_file = os.path.join('/Users/luismartins/Desktop/Tese/NexIK_Flexibility/Forecast_24h/Real Values - FC24 Cost', "results.xlsx")

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df_X.to_excel(writer, sheet_name="X", index=False)
    df_X_opt.to_excel(writer, sheet_name="X_Opt", index=False)
    df_W.to_excel(writer, sheet_name="Weights", index=False)
    df_W_base.to_excel(writer, sheet_name="Weights_Base", index=False)

print("Excel criado em:")
print(output_file)

Excel criado em:
/Users/luismartins/Desktop/Tese/NexIK_Flexibility/Forecast_24h/Real Values - FC24 Cost/results.xlsx
